In [1]:
# -*- coding: utf-8 -*-
import math
import numpy as np
import mlx.core as mx
import mlx.nn as nn

# =========================================================================
# 1. 組態與工具函數
# =========================================================================

class Mamba3Config:
    def __init__(
        self, d_model=768, d_state=64, d_head=64, n_groups=1, mimo_rank=4, expand=4,
        num_layers=15, use_kmoe=True,
        kmoe_num_experts=8, kmoe_top_k=2, kmoe_r1=4, kmoe_r2=1024, kmoe_r3=256, 
        ffn_expand=6, num_kv_heads=4, rms_norm_eps=1e-5, layer_scale_init=1e-2
    ):
        self.d_model = d_model
        self.d_state = d_state
        self.d_head = d_head
        self.expand = expand
        self.num_layers = num_layers
        self.d_inner = int(expand * d_model)
        self.n_heads = self.d_inner // d_head
        self.n_groups = n_groups
        self.mimo_rank = mimo_rank
        self.rms_norm_eps = rms_norm_eps
        
        self.use_kmoe = use_kmoe
        self.kmoe_num_experts = kmoe_num_experts
        self.kmoe_top_k = kmoe_top_k
        self.kmoe_r1 = kmoe_r1
        self.kmoe_r2 = kmoe_r2
        self.kmoe_r3 = kmoe_r3
        self.ffn_expand = ffn_expand
        
        self.num_kv_heads = num_kv_heads
        self.kv_groups = self.n_heads // num_kv_heads
        self.layer_scale_init = layer_scale_init

class LayerScale(nn.Module):
    def __init__(self, dim, init_value=1e-2):
        super().__init__()
        self.gamma = mx.array(np.ones(dim) * init_value, dtype=mx.float32)

    def __call__(self, x):
        return x * self.gamma.astype(x.dtype)

def fast_scaled_tanh(x, scale=10.0):
    return mx.tanh(x / scale) * scale

def silu(x):
    return x * mx.sigmoid(x)

def fast_silu_gating(gate, feat):
    return silu(gate) * feat

# =========================================================================
# 2. TuckerMoE & FeedForward (MLX 版)
# =========================================================================

class TuckerMoE(nn.Module):
    def __init__(self, dim_in, dim_out, num_experts=8, top_k=2, r1=4, r2=1024, r3=256):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = min(top_k, num_experts)
        
        self.router = nn.Linear(dim_in, num_experts, bias=False)
        
        # MLX 不像 PyTorch 有 nn.Parameter，直接宣告陣列即可
        self.U_expert = mx.random.normal((num_experts, r1)) * 0.02
        self.U_in = mx.random.normal((dim_in, r3)) * 0.02
        self.U_out = mx.random.normal((r2, dim_out)) * 0.02
        self.core = mx.random.normal((r1, r3, r2)) * 0.02
        self.bias = mx.zeros((dim_out,))

        self.inner_norm = nn.RMSNorm(r3)

    def __call__(self, x, temperature=0.5):
        orig_shape = x.shape
        x_flat = x.reshape(-1, orig_shape[-1])
        
        # Routing
        raw_logits = self.router(x_flat)
        capped = fast_scaled_tanh(raw_logits, 10.0)
        router_logits = capped / temperature
        
        # 修正區塊：使用 argsort 取得 indices，再 take 取得 values
        # 因為加上負號排序，所以前 k 個就是最大值
        top_k_indices = mx.argsort(-router_logits, axis=-1)[..., :self.top_k]
        top_k_probs = mx.take_along_axis(router_logits, top_k_indices, axis=-1)
        
        top_k_probs = mx.softmax(top_k_probs, axis=-1) # (B, K)
        
        # Tucker Core Logic
        x_shared = self.inner_norm(mx.matmul(x_flat, self.U_in)) # (B, r3)
        
        # G_experts: (E, r3, r2)
        G_experts = mx.einsum('er, rst -> est', self.U_expert, self.core)
        
        # Gather experts for each token: (B, K, r3, r2)
        G_token = G_experts[top_k_indices]
        
        # Multiply shared features with experts: (B, K, r2)
        expert_outs = mx.einsum('br, bkrt -> bkt', x_shared, G_token)
        
        # Weight by probs: (B, r2)
        out_core = mx.sum(expert_outs * mx.expand_dims(top_k_probs, -1), axis=1)
        
        out = mx.matmul(out_core, self.U_out).reshape(*orig_shape[:-1], -1)
        return out + self.bias

class MixtralMoEFeedForward(nn.Module):
    def __init__(self, config: Mamba3Config):
        super().__init__()
        d_ff = int(math.ceil(config.ffn_expand * config.d_model / 256) * 256)
        kw = dict(num_experts=config.kmoe_num_experts, top_k=config.kmoe_top_k,
                  r1=config.kmoe_r1, r2=config.kmoe_r2, r3=config.kmoe_r3)
        self.gate_proj = TuckerMoE(config.d_model, d_ff, **kw)
        self.up_proj   = TuckerMoE(config.d_model, d_ff, **kw)
        self.down_proj = TuckerMoE(d_ff, config.d_model, **kw)

    def __call__(self, x):
        gate = self.gate_proj(x)
        feat = self.up_proj(x)
        y = self.down_proj(fast_silu_gating(gate, feat))
        return y

# =========================================================================
# 3. Mamba3 區塊 & 轉換
# =========================================================================

class Mamba3Block(nn.Module):
    def __init__(self, config: Mamba3Config):
        super().__init__()
        self.config = config
        d_in, H, G, P, N, R = config.d_model, config.n_heads, config.n_groups, config.d_head, config.d_state, config.mimo_rank
        self.ratio, self.dim_z, self.dim_x = H // G, H * P, H * P
        self.dim_B, self.dim_C, self.dim_dt, self.dim_A, self.dim_lambda = G*N*R, G*N*R, G, G, G
        
        total_in_dim = self.dim_z + self.dim_x + self.dim_B + self.dim_C + self.dim_dt + self.dim_A + self.dim_lambda
        self.in_proj = nn.Linear(d_in, total_in_dim, bias=True)
        
        if config.use_kmoe:
            kw = dict(num_experts=config.kmoe_num_experts, top_k=config.kmoe_top_k, 
                      r1=config.kmoe_r1, r2=config.kmoe_r2, r3=config.kmoe_r3)
            self.x_up_proj = TuckerMoE(H*P, H*P*R, **kw)
            self.out_proj  = TuckerMoE(d_in, d_in, **kw)
        else:
            self.x_up_proj = nn.Linear(P, P*R, bias=False)
            self.out_proj  = nn.Linear(d_in, d_in, bias=False)
            
        self.y_down_proj      = nn.Linear(P*R, P, bias=False)
        self.theta_log        = mx.random.normal((G, N//2))
        self.D                = mx.ones((H,))
        self.norm_B           = nn.RMSNorm(N*R, eps=config.rms_norm_eps)
        self.norm_C           = nn.RMSNorm(N*R, eps=config.rms_norm_eps)
        self.bias_B           = mx.zeros((G, N, R))
        self.bias_C           = mx.zeros((G, N, R))
        self.mamba_dense_proj = nn.Linear(config.d_inner, d_in, bias=False)
        self.pre_gate_norm    = nn.RMSNorm(H*P)
        self.norm_mamba       = nn.RMSNorm(config.d_model)
        self.norm_out_proj    = nn.RMSNorm(config.d_model)
        self.ls_mamba         = LayerScale(config.d_model, init_value=config.layer_scale_init)
        self.ls_out_proj      = LayerScale(config.d_model, init_value=config.layer_scale_init)

    def apply_rope(self, x, angles):
        N_half = angles.shape[-1]
        x_reshaped = x.reshape(*x.shape[:-2], N_half, 2, x.shape[-1])
        x1, x2 = x_reshaped[..., 0, :], x_reshaped[..., 1, :]
        sin_a = mx.expand_dims(mx.sin(angles), -1)
        cos_a = mx.expand_dims(mx.cos(angles), -1)
        out = mx.stack([x1 * cos_a - x2 * sin_a, x2 * cos_a + x1 * sin_a], axis=-2)
        return out.reshape(x.shape)

    def __call__(self, x):
        B_sz, L, _ = x.shape
        H, G, P, N, R, ratio = self.config.n_heads, self.config.n_groups, self.config.d_head, self.config.d_state, self.config.mimo_rank, self.ratio
        
        residual_mamba = x
        u = self.norm_mamba(x)
        
        proj_out = self.in_proj(u)
        splits = [self.dim_z, self.dim_x, self.dim_B, self.dim_C, self.dim_dt, self.dim_A, self.dim_lambda]
        split_indices = np.cumsum(splits)[:-1].tolist()
        z, x_prime, B_param, C_param, dt, A_param, lambda_param = mx.split(proj_out, split_indices, axis=-1)
        
        x_prime = x_prime.reshape(B_sz, L, H, P)
        
        # Softplus 替代
        dt = mx.logaddexp(mx.array(0.0), dt)
        A = -mx.exp(A_param)
        theta = mx.exp(self.theta_log)
        
        def bg(t):
            return mx.repeat(t, ratio, axis=2)
            
        dt_b = bg(mx.expand_dims(dt, -1)).squeeze(-1)
        A_b = bg(mx.expand_dims(A, -1)).squeeze(-1)
        
        theta_rep = mx.repeat(theta, ratio, axis=0)
        angles = mx.cumsum(mx.einsum("blh, hn -> blhn", dt_b, theta_rep), axis=1)
        
        B_reshaped = self.norm_B(B_param.reshape(B_sz, L, G, N*R)).reshape(B_sz, L, G, N, R)
        C_reshaped = self.norm_C(C_param.reshape(B_sz, L, G, N*R)).reshape(B_sz, L, G, N, R)
        
        B_rotated = self.apply_rope(bg(B_reshaped) + self.bias_B, angles)
        C_rotated = self.apply_rope(bg(C_reshaped) + self.bias_C, angles)
        
        if self.config.use_kmoe:
            x_up = self.x_up_proj(x_prime.reshape(B_sz, L, -1))
            x_ssm = x_up.reshape(B_sz, L, H, P, R)
        else:
            x_ssm = self.x_up_proj(x_prime).reshape(B_sz, L, H, P, R)
            
        input_signal = mx.einsum("blhnr, blhpr -> blhnp", B_rotated, x_ssm)
        lv = mx.sigmoid(bg(mx.expand_dims(lambda_param, -1)).squeeze(-1)).reshape(B_sz, L, H, 1, 1)
        dv = dt_b.reshape(B_sz, L, H, 1, 1)
        av = mx.exp(dt_b * A_b).reshape(B_sz, L, H, 1, 1)
        
        # input_signal roll
        ip = mx.concatenate([mx.zeros_like(input_signal[:, :1]), input_signal[:, :-1]], axis=1)
        u_ssm = lv * dv * input_signal + (1 - lv) * dv * av * ip

        # MLX 的序列掃描 (取代 Triton Parallel Scan)
        h_s = mx.zeros((B_sz, H, N, P), dtype=x.dtype)
        y_list = []
        for t in range(L):
            h_s = h_s * av[:, t] + u_ssm[:, t]
            y_t = mx.einsum("bhnp, bhnr -> bhpr", h_s, C_rotated[:, t])
            y_list.append(y_t)
            
        y_stack = mx.stack(y_list, axis=1)
        
        y = self.y_down_proj(y_stack.reshape(B_sz, L, H, P*R)).reshape(B_sz, L, H*P)
        D_rep = mx.repeat(self.D, P, axis=0)
        y = y + x_prime.reshape(B_sz, L, H*P) * D_rep
        
        mamba_out = self.mamba_dense_proj(self.pre_gate_norm(y) * silu(z))
        mid_x = residual_mamba + self.ls_mamba(mamba_out)
        
        normed_mid = self.norm_out_proj(mid_x)
        proj_out = self.out_proj(normed_mid) if self.config.use_kmoe else self.out_proj(normed_mid)
        
        return mid_x + self.ls_out_proj(proj_out)

# =========================================================================
# 4. Transformer Block & 主模型
# =========================================================================

class TransformerBlock(nn.Module):
    def __init__(self, config: Mamba3Config):
        super().__init__()
        self.head_dim = 64
        self.num_heads = config.d_model // 64
        self.num_kv_heads = config.num_kv_heads
        self.kv_groups = self.num_heads // config.num_kv_heads
        
        self.q_proj = nn.Linear(config.d_model, self.num_heads * 64, bias=False)
        self.k_proj = nn.Linear(config.d_model, self.num_kv_heads * 64, bias=False)
        self.v_proj = nn.Linear(config.d_model, self.num_kv_heads * 64, bias=False)
        self.o_proj = nn.Linear(config.d_model, config.d_model, bias=True)
        
        self.norm_attn = nn.RMSNorm(config.d_model)
        self.use_kmoe = config.use_kmoe
        
        if config.use_kmoe:
            self.ffn = MixtralMoEFeedForward(config)
        else:
            d_ff = int(math.ceil(8 * config.d_model / 3 / 256) * 256)
            self.ffn_gate = nn.Linear(config.d_model, d_ff, bias=False)
            self.ffn_up   = nn.Linear(config.d_model, d_ff, bias=False)
            self.ffn_down = nn.Linear(d_ff, config.d_model, bias=False)
            
        self.norm_ffn = nn.RMSNorm(config.d_model)
        self.ls_attn = LayerScale(config.d_model, init_value=config.layer_scale_init)
        self.ls_ffn  = LayerScale(config.d_model, init_value=config.layer_scale_init)

    def __call__(self, x):
        B, L, D = x.shape
        residual = x
        nx = self.norm_attn(x)
        
        q = self.q_proj(nx).reshape(B, L, self.num_heads, 64).transpose(0, 2, 1, 3)
        k = self.k_proj(nx).reshape(B, L, self.num_kv_heads, 64).transpose(0, 2, 1, 3)
        v = self.v_proj(nx).reshape(B, L, self.num_kv_heads, 64).transpose(0, 2, 1, 3)
        
        # GQA Expand
        k = mx.repeat(k, self.kv_groups, axis=1)
        v = mx.repeat(v, self.kv_groups, axis=1)
        
        # MLX 內建的高效注意力機制
        attn = mx.fast.scaled_dot_product_attention(q, k, v, scale=1.0/math.sqrt(64))
        
        x = residual + self.ls_attn(self.o_proj(attn.transpose(0, 2, 1, 3).reshape(B, L, D)))
        
        residual = x
        h = self.norm_ffn(x)
        
        if self.use_kmoe:
            ffn_out = self.ffn(h)
        else:
            ffn_out = self.ffn_down(fast_silu_gating(self.ffn_gate(h), self.ffn_up(h)))
            
        return residual + self.ls_ffn(ffn_out)

class TrueHybridMamba(nn.Module):
    def __init__(self, config: Mamba3Config, mamba_ratio=4):
        super().__init__()
        self.layers = []
        for _ in range(config.num_layers):
            for _ in range(mamba_ratio):
                self.layers.append(Mamba3Block(config))
            self.layers.append(TransformerBlock(config))

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

class Mamba3LanguageModel(nn.Module):
    def __init__(self, config: Mamba3Config, vocab_size: int):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(vocab_size, config.d_model)
        self.backbone = TrueHybridMamba(config)
        self.norm = nn.RMSNorm(config.d_model)
        # Weight tying
        self.head = nn.Linear(config.d_model, vocab_size, bias=False)

    def __call__(self, input_ids):
        # Tie weights implicitly during forward
        self.head.weight = self.embed.weight
        
        x = self.embed(input_ids)
        hidden = self.backbone(x)
        hidden = self.norm(hidden)
        
        logits = fast_scaled_tanh(self.head(hidden / math.sqrt(self.config.d_model)), 30.0)
        return logits


In [ ]:
import time
import argparse
import mlx.core as mx

# =========================================================================
# 進階取樣器 (Top-K, Top-P, Min-P, 懲罰機制)
# =========================================================================
def apply_penalties(logits, generated_tokens, args):
    """套用重複性、存在性與頻率懲罰"""
    if not generated_tokens:
        return logits
    
    # 計算每個 token 出現的頻率
    counts = {}
    for t in generated_tokens:
        counts[t] = counts.get(t, 0) + 1

    # 將 logits 轉為 numpy 處理這些離散且複雜的索引會比較直觀，再轉回 mx.array
    # (在極度追求效能時，這裡可以用純 MLX array 操作，但為了支援所有懲罰且避免動態圖變大，稍微混合操作)
    logits_np = np.array(logits)
    
    for t, count in counts.items():
        # Repetition Penalty (重複懲罰)
        if args.rep_pen != 1.0:
            if logits_np[t] > 0:
                logits_np[t] /= args.rep_pen
            else:
                logits_np[t] *= args.rep_pen
        
        # Presence Penalty (存在懲罰) & Frequency Penalty (頻率懲罰)
        logits_np[t] -= (args.pres_pen + count * args.freq_pen)
        
    return mx.array(logits_np)
def advanced_sample(logits, args):
    """整合 Temp, Top-K, Top-P, Min-P 的取樣函數"""
    if args.temp == 0.0:
        return mx.argmax(logits, axis=-1)

    logits = logits / args.temp
    probs = mx.softmax(logits, axis=-1)
    
    # Min-P 處理 (MLX 原生 mx.where 支援得很好)
    if args.min_p > 0.0:
        p_max = mx.max(probs)
        logits = mx.where(probs < (args.min_p * p_max), -1e9, logits)
        probs = mx.softmax(logits, axis=-1)

    # Top-K 處理
    if args.top_k > 0:
        top_k_indices = mx.argsort(-logits)
        kth_val = logits[top_k_indices[args.top_k - 1]]
        logits = mx.where(logits < kth_val, -1e9, logits)
        probs = mx.softmax(logits, axis=-1)

    # Top-P 處理
    if args.top_p < 1.0:
        # 🎯 終極修正：MLX 目前對布林索引支援不佳，我們把 Top-P 邏輯完全交給 NumPy
        probs_np = np.array(probs)
        logits_np = np.array(logits)
        
        sorted_indices = np.argsort(-probs_np)
        sorted_probs = probs_np[sorted_indices]
        cumulative_probs = np.cumsum(sorted_probs, axis=-1)
        
        # 運用 NumPy 自由的遮罩與平移操作
        mask = cumulative_probs > args.top_p
        shifted_mask = np.concatenate([[False], mask[:-1]])
        
        # NumPy 完美支援布林索引
        indices_to_remove = sorted_indices[shifted_mask]
        logits_np[indices_to_remove] = -1e9
        
        # 處理完再轉回 MLX array
        logits = mx.array(logits_np)

    # 最終抽樣
    next_token = mx.random.categorical(logits)
    return next_token

# =========================================================================
# 推論主函數
# =========================================================================
def generate(model, prompt_tokens, args):
    print(f"🧠 開始生成 (提示詞長度: {len(prompt_tokens)} tokens)...")
    
    # JIT 編譯前向傳遞 (Step)
    # ⚠️ 注意：由於我們目前沒有實作 KV-Cache 和 SSM State 緩存，
    # 每次 x 的長度增加都會觸發重新編譯。為了展示 @mx.compile 的用法先保留，
    # 若發現長度增長時卡頓，可以將 @mx.compile 註解掉，或未來實作固定長度的 State Cache。
    @mx.compile
    def forward_step(x):
        return model(x)[0, -1, :]

    current_tokens = prompt_tokens.copy()
    generated_new_tokens = []
    
    start_time = time.time()
    
    for i in range(args.max_tokens):
        # 準備輸入 (統一記憶體：直接將 Python list 轉 mx.array，無需 .to(device))
        x = mx.array([current_tokens])
        
        # 取得最後一個 Token 的 logits
        logits = forward_step(x)
        
        # 1. 套用存在/頻率/重複懲罰
        logits = apply_penalties(logits, generated_new_tokens, args)
        
        # 2. 進行進階取樣
        next_token_arr = advanced_sample(logits, args)
        
        # 3. 懶惰計算 (Lazy Evaluation) 核心：
        # 強制評估這個 token，避免 MLX 在背景建立過大的未執行計算圖，導致 OOM 或變慢
        mx.eval(next_token_arr)
        
        next_token = next_token_arr.item()
        current_tokens.append(next_token)
        generated_new_tokens.append(next_token)
        
        print(next_token, end=" ", flush=True)
        
    end_time = time.time()
    elapsed = end_time - start_time
    tps = args.max_tokens / elapsed
    print(f"\n✨ 生成完成！總共 {args.max_tokens} tokens，耗時 {elapsed:.2f} 秒 ({tps:.2f} tokens/s)")
    return current_tokens


if __name__ == "__main__":
    # 使用 argparse 設定所有參數
    parser = argparse.ArgumentParser(description="Mamba3 MLX Inference")
    parser.add_argument("--max_tokens", type=int, default=512, help="最大生成長度")
    parser.add_argument("--temp", type=float, default=0.5, help="溫度 (Temperature)")
    parser.add_argument("--top_k", type=int, default=40, help="Top-K 採樣")
    parser.add_argument("--top_p", type=float, default=0.95, help="Top-P 採樣")
    parser.add_argument("--min_p", type=float, default=0.05, help="Min-P 採樣")
    parser.add_argument("--rep_pen", type=float, default=1.2, help="重複懲罰 (Repetition Penalty)")
    parser.add_argument("--pres_pen", type=float, default=0.0, help="存在懲罰 (Presence Penalty)")
    parser.add_argument("--freq_pen", type=float, default=0.15, help="頻率懲罰 (Frequency Penalty)")
    args = parser.parse_args()

    # 初始化參數
    config = Mamba3Config(
        d_model=768, d_state=64, d_head=64, expand=2, num_layers=6,
        mimo_rank=4, num_kv_heads=4, use_kmoe=True,
        kmoe_num_experts=8, kmoe_top_k=2, kmoe_r1=32, kmoe_r2=512, kmoe_r3=256,
        ffn_expand=6
    )
    VOCAB_SIZE = 32007

    print("🚀 正在 Apple Silicon 上初始化 Mamba3 Hybrid TuckerMoE (MLX)...")
    model = Mamba3LanguageModel(config, VOCAB_SIZE)
    
    # 強制評估模型權重，載入記憶體
    mx.eval(model.parameters()) 

    # 模擬 Prompt
    dummy_prompt = [101, 2023, 1045, 2066, 2186]  
    
    # 進行推理，將 argparse 的變數傳入
    generate(model, dummy_prompt, args)

usage: ipykernel_launcher.py [-h] [--max_tokens MAX_TOKENS] [--temp TEMP]
                             [--top_k TOP_K] [--top_p TOP_P] [--min_p MIN_P]
                             [--rep_pen REP_PEN] [--pres_pen PRES_PEN]
                             [--freq_pen FREQ_PEN]
ipykernel_launcher.py: error: argument --freq_pen: invalid float value: '/Users/hungwei/Library/Jupyter/runtime/kernel-v3b64894c7e0814deea67b874848381119cd7a65f6.json'


SystemExit: 2

/Users/hungwei/Desktop/Proj/Mamba3-XR/.venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
